# 🏭 PhARMA RAG — Scale Edition
### The same RAG pipeline, rebuilt for *hundreds* of real drugs downloaded live from openFDA

This is the big-scale sequel to **PhARMA RAG — Learning Edition**. Same teaching rhythm —

> **🧠 Concept** → **💻 Code** → **🔍 What just happened**

— but now every step is stressed by **real, messy, large data**, which forces new ideas that the toy
12-drug version never needed:

| New at scale | Why it appears now |
|:--|:--|
| **Live data download** (openFDA API) | 12 hand-written drugs → *hundreds* of real drug labels |
| **Sliding-window chunking** | real fields are long paragraphs, not one tidy sentence |
| **Batched + disk-cached embeddings** | encoding thousands of chunks is slow — do it once, save it |
| **FAISS approximate index (IVF)** | brute-force cosine stops being free at thousands of vectors |
| **Auto-generated evaluation set** | you can't hand-label ground truth for hundreds of drugs |
| **Cross-encoder reranking** | cheap retrieval casts a wide net; a reranker sharpens the top-k |

Everything the Learning Edition taught still holds — **medically-safe preprocessing, sparse/dense/
hybrid retrieval, the grounded prompt, and the refusal test**. We're adding the *engineering* that
makes it survive scale.

> ▶️ **Run order:** `Cell → Run All`. The notebook downloads data on first run and **caches it to
> disk**, so later runs are fast and work offline. Every optional dependency (embeddings, FAISS,
> reranker, Ollama) degrades gracefully with a clear message.

---
# Part 0 — Setup & Configuration

In [ ]:
# Uncomment once if needed (internet required):
# !pip install -q pandas numpy scikit-learn rank-bm25 sentence-transformers faiss-cpu nltk requests

In [ ]:
import json, os, re, math, time, warnings, urllib.request
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

# ── Configuration ────────────────────────────────────────────────────
RANDOM_SEED      = 42
TARGET_DRUGS     = 500                     # how many unique drugs to download
K                = 5                       # final results shown to the LLM
RETRIEVE_POOL    = 50                      # wide net retrieved before reranking
CHUNK_WORDS      = 90                      # sliding-window chunk size (words)
CHUNK_OVERLAP    = 25                      # words shared between neighbouring chunks
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
RERANK_MODEL     = "cross-encoder/ms-marco-MiniLM-L-6-v2"
OLLAMA_HOST      = "http://localhost:11434"
OLLAMA_MODEL     = "deepseek-r1:1.5b"

np.random.seed(RANDOM_SEED)

DATA_DIR   = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)
DRUGS_LARGE = DATA_DIR / "drugs_large.json"      # downloaded dataset (cached)
EMB_CACHE   = DATA_DIR / "doc_embeddings.npy"    # cached embeddings
print("Config loaded. Data dir:", DATA_DIR)

In [ ]:
# ── NLTK resources (with graceful fallback) ──
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass
print("NLTK ready (fallbacks exist if any download failed).")

---
# Part 1 — Downloading Real Data at Scale (openFDA)

## 1.1 Fetch drug labels from the openFDA API

### 🧠 Concept
The toy notebook shipped 12 hand-written drugs. Real RAG systems ingest data from **live sources**.
[openFDA](https://open.fda.gov) exposes ~260,000 official FDA drug labels as JSON — free, no API key.

We page through the API, **keep only records rich enough to be useful** (they must have indications,
mechanism, and dosage), **deduplicate by generic name**, and **normalize** each label into the exact
same schema as the toy `drugs.json`. Crucially we **cache the result to disk** — download once, then
every rerun is instant and offline-safe. That download-once-cache pattern is standard at scale.

In [ ]:
SEARCH = ("_exists_:indications_and_usage+AND+_exists_:mechanism_of_action"
          "+AND+_exists_:dosage_and_administration")
FIELD_MAP = {  # our schema field : openFDA field
    "indications": "indications_and_usage", "mechanism": "mechanism_of_action",
    "dosage": "dosage_and_administration",  "side_effects": "adverse_reactions",
    "contraindications": "contraindications", "interactions": "drug_interactions",
    "description": "description",
}
MAX_CHARS = 4000

def _clean(v):
    if isinstance(v, list): v = " ".join(str(x) for x in v)
    return re.sub(r"\s+", " ", str(v)).strip()[:MAX_CHARS]

def download_openfda(target=TARGET_DRUGS, per_page=100):
    seen, drugs, skip = set(), [], 0
    while len(drugs) < target and skip < 25000:
        url = (f"https://api.fda.gov/drug/label.json?search={SEARCH}"
               f"&limit={per_page}&skip={skip}")
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                batch = json.load(r).get("results", [])
        except Exception as e:
            print(f"  stop at skip={skip}: {type(e).__name__}"); break
        if not batch: break
        for rec in batch:
            of = rec.get("openfda", {})
            name = (of.get("generic_name") or of.get("brand_name") or [""])[0].strip().title()
            if not name or name.lower() in seen: continue
            if not rec.get("indications_and_usage") or not rec.get("mechanism_of_action"): continue
            cat = (of.get("pharm_class_epc") or of.get("pharm_class_moa")
                   or of.get("route") or ["Unclassified"])[0].title()
            d = {"name": name, "category": cat}
            for our, fda in FIELD_MAP.items():
                d[our] = _clean(rec.get(fda, ""))
            if not d["description"]: d["description"] = d["indications"][:600]
            d["source"] = "openFDA drug label (SPL / DailyMed)"
            if sum(1 for f in FIELD_MAP if d[f]) >= 4:
                seen.add(name.lower()); drugs.append(d)
        skip += per_page
        print(f"  skip={skip:>5}  collected={len(drugs)}")
    return drugs

# Download once, then reuse the cached file
if DRUGS_LARGE.exists():
    drugs_data = json.load(open(DRUGS_LARGE, encoding="utf-8"))
    print(f"Loaded cached dataset: {len(drugs_data)} drugs  ({DRUGS_LARGE})")
else:
    print("Downloading from openFDA (first run only)...")
    drugs_data = download_openfda()
    json.dump(drugs_data, open(DRUGS_LARGE, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"Saved {len(drugs_data)} drugs → {DRUGS_LARGE}")

In [ ]:
# Peek at what we got
print("Total drugs:", len(drugs_data))
cats = Counter(d["category"] for d in drugs_data)
print("\nTop 10 pharmacologic classes:")
for c, n in cats.most_common(10):
    print(f"  {n:>4}  {c}")
print("\nExample drug:", drugs_data[0]["name"], "—", drugs_data[0]["category"])
print("Indications snippet:", drugs_data[0]["indications"][:160], "...")

### 🔍 What just happened
We turned a live API into a clean, on-disk corpus of real drugs spanning many pharmacologic classes.
Note the text is **long and messy** (real regulatory language), which is exactly why the next step —
chunking — has to change.

---
# Part 2 — Chunking at Scale + Safe Preprocessing

## 2.1 Why field-level chunking breaks on real data

### 🧠 Concept
In the toy set, one field = one short sentence, so "one chunk per field" was fine. Real openFDA
fields are **long multi-paragraph blocks** (hundreds of words). A single giant chunk is bad for
retrieval because:
- the embedding of a 600-word block is a blurry average — it matches everything weakly;
- the answer sentence gets diluted by surrounding text.

**The fix: sliding-window chunking.** Split long text into overlapping windows of ~90 words
(25-word overlap). Overlap prevents a fact from being sliced in half at a boundary. Each window
keeps its drug/field/category **metadata** so we can still cite precisely.

In [ ]:
def sliding_windows(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    """Split text into overlapping word windows. Short text → a single chunk."""
    words = text.split()
    if len(words) <= size:
        return [text] if words else []
    step, out, i = size - overlap, [], 0
    while i < len(words):
        out.append(" ".join(words[i:i + size]))
        if i + size >= len(words):
            break
        i += step
    return out

FIELD_LABELS = ["description", "mechanism", "indications", "dosage",
                "side_effects", "contraindications", "interactions"]

records = []
for drug in drugs_data:
    for field in FIELD_LABELS:
        body = drug.get(field, "")
        if not body:
            continue
        for j, window in enumerate(sliding_windows(body)):
            records.append({
                "drug": drug["name"].lower(),
                "category": drug["category"].lower(),
                "field": field,
                "win": j,
                "text": f"{drug['name']} — {field}: {window}",
            })

df_chunks = pd.DataFrame(records).reset_index(drop=True)
print(f"{len(drugs_data)} drugs → {len(df_chunks)} chunks "
      f"(avg {len(df_chunks)/len(drugs_data):.1f} chunks/drug)")
df_chunks["word_count"] = df_chunks["text"].str.split().str.len()
print("Chunk word-count:", df_chunks["word_count"].describe()[["mean","min","max"]].round(1).to_dict())
df_chunks.head(3)

### 🔍 What just happened
The corpus jumped from ~84 chunks to **thousands** — and every chunk is now a tight, uniformly-sized
window instead of a whole field. This is the single biggest reason the retrieval below behaves better
than a naive large-blob approach.

## 2.2 Medically-safe preprocessing (unchanged philosophy)

### 🧠 Concept
Scale doesn't change the safety rule from the Learning Edition: **never delete numbers or negation**.
`no known interactions` and `500 mg` must survive. We reuse the same modular cleaner and the
`readable_lemmatized` profile as the default for lexical retrieval.

In [ ]:
import string
def safe_tok(t):
    try: return nltk.word_tokenize(t)
    except Exception: return t.split()
try: _STOP = set(nltk.corpus.stopwords.words("english"))
except Exception:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS; _STOP = set(ENGLISH_STOP_WORDS)
from nltk.stem import PorterStemmer, WordNetLemmatizer
_STEM, _LEM = PorterStemmer(), WordNetLemmatizer()
def _lem(t):
    try: return _LEM.lemmatize(t)
    except Exception: return t
PROTECTED_NEGATION = {"no","not","nor","never","none","neither","nobody","nothing","nowhere"}

def preprocess_text(text, lowercase=True, remove_url=True, remove_punct=False, remove_num=False,
                    normalize_space=True, remove_stop_words=False, preserve_negation=True,
                    use_stemming=False, use_lemmatization=False):
    if not isinstance(text, str) or not text.strip(): return ""
    if remove_url: text = re.sub(r"https?://\S+", "", text)
    if lowercase: text = text.lower()
    if remove_punct: text = text.translate(str.maketrans("", "", string.punctuation))
    if remove_num: text = re.sub(r"\b\d+\b", "", text)
    if normalize_space: text = re.sub(r"\s+", " ", text).strip()
    if remove_stop_words or use_stemming or use_lemmatization:
        toks = safe_tok(text)
        if remove_stop_words:
            toks = [t for t in toks if t not in _STOP or (preserve_negation and t in PROTECTED_NEGATION)]
        if use_stemming:
            toks = [_STEM.stem(t) if t not in PROTECTED_NEGATION else t for t in toks]
        if use_lemmatization:
            toks = [_lem(t) if t not in PROTECTED_NEGATION else t for t in toks]
        text = " ".join(toks)
    return re.sub(r"\s+", " ", text).strip() if normalize_space else text

DEFAULT_PROFILE = dict(lowercase=True, remove_url=True, normalize_space=True,
                       remove_stop_words=True, preserve_negation=True, use_lemmatization=True)

t0 = time.time()
df_chunks["clean"] = df_chunks["text"].apply(lambda t: preprocess_text(t, **DEFAULT_PROFILE))
print(f"Preprocessed {len(df_chunks)} chunks in {time.time()-t0:.1f}s")
print("Safety check:", preprocess_text("No known interactions at 500 mg", **DEFAULT_PROFILE))

---
# Part 3 — Auto-Generated Evaluation at Scale

## 3.1 You can't hand-label hundreds of drugs

### 🧠 Concept
The Learning Edition wrote 10 ground-truth queries by hand. That doesn't scale to hundreds of drugs.
Instead we **generate the evaluation set programmatically**: sample N random drugs, and for each build
a templated question whose correct answer is *known by construction* — the chunks belonging to that
drug's relevant field.

e.g. for a sampled drug `X`: *"What are the side effects of X?"* → ground truth = all chunks where
`drug == X and field == side_effects`. This gives us a large, unbiased, reproducible eval set.

In [ ]:
EVAL_TEMPLATES = [
    ("side_effects",      "What are the side effects of {d}?"),
    ("mechanism",         "How does {d} work?"),
    ("indications",       "What is {d} used for?"),
    ("dosage",            "What is the dosage of {d}?"),
    ("contraindications", "When should {d} not be used?"),
    ("interactions",      "What does {d} interact with?"),
]

rng = np.random.default_rng(RANDOM_SEED)
# Only sample drugs that actually have the target field (so ground truth is non-empty)
field_index = {f: df_chunks[df_chunks["field"] == f]["drug"].unique() for f, _ in EVAL_TEMPLATES}

QUERY_SPECS, N_PER_TEMPLATE = [], 10
for field, template in EVAL_TEMPLATES:
    pool = field_index[field]
    for d in rng.choice(pool, size=min(N_PER_TEMPLATE, len(pool)), replace=False):
        gt = set(df_chunks[(df_chunks["drug"] == d) & (df_chunks["field"] == field)].index)
        QUERY_SPECS.append({"query": template.format(d=d.title()), "gt_indices": gt})

print(f"Generated {len(QUERY_SPECS)} evaluation queries with known ground truth.")
for s in QUERY_SPECS[:4]:
    print(f'  "{s["query"]}"  → {len(s["gt_indices"])} correct chunk(s)')

## 3.2 Metrics + evaluation harness (same as before)

### 🧠 Concept
Identical metrics to the Learning Edition — **Precision@k, Recall@k, Hit@k, MRR** — because the
questions "is it precise / complete / did it hit / how high" don't change with scale. One harness
scores every retriever.

In [ ]:
def precision_at_k(r, gt, k): return 0.0 if k == 0 else len(set(r[:k]) & gt) / k
def recall_at_k(r, gt, k):    return 0.0 if not gt else len(set(r[:k]) & gt) / len(gt)
def hit_at_k(r, gt, k):       return 1.0 if set(r[:k]) & gt else 0.0
def rr(r, gt):
    for i, idx in enumerate(r):
        if idx in gt: return 1.0 / (i + 1)
    return 0.0

def evaluate_retriever(fn, specs=None, k=K):
    specs = specs or QUERY_SPECS
    P = R = H = M = 0.0
    for s in specs:
        got = [idx for idx, _ in fn(s["query"], k=k)]
        gt = s["gt_indices"]
        P += precision_at_k(got, gt, k); R += recall_at_k(got, gt, k)
        H += hit_at_k(got, gt, k);       M += rr(got, gt)
    n = len(specs)
    return {"P@k": P/n, "R@k": R/n, "Hit": H/n, "MRR": M/n}

print(f"Harness ready — evaluates over {len(QUERY_SPECS)} queries.")

---
# Part 4 — Sparse Retrieval at Scale (TF-IDF, BM25)

## 4.1 TF-IDF and BM25 over thousands of chunks

### 🧠 Concept
The sparse methods scale gracefully — the TF-IDF matrix is **sparse** (mostly zeros), so thousands of
chunks stay cheap in memory. Same golden rule: **documents → `fit_transform`, query → `transform`**.
BM25 again uses `rank_bm25` if present, else our compact NumPy fallback. We also **time** them —
timing is a first-class metric at scale.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)     # min_df=2 drops one-off noise at scale
tfidf_matrix = tfidf_vec.fit_transform(df_chunks["clean"])
print(f"TF-IDF matrix: {tfidf_matrix.shape}  (sparse, {tfidf_matrix.nnz:,} non-zeros)")

def retrieve_tfidf(query, k=5):
    q = tfidf_vec.transform([preprocess_text(query, **DEFAULT_PROFILE)])
    sc = cosine_similarity(q, tfidf_matrix).flatten()
    idx = sc.argsort()[::-1][:k]
    return [(int(i), float(sc[i])) for i in idx if sc[i] > 0]

t0 = time.time(); tfidf_summary = evaluate_retriever(retrieve_tfidf); dt = time.time() - t0
print(f"TF-IDF  { {k: round(v,3) for k,v in tfidf_summary.items()} }  ({dt:.1f}s for {len(QUERY_SPECS)} queries)")

In [ ]:
# BM25 with self-contained NumPy fallback
try:
    from rank_bm25 import BM25Okapi; BM25_SRC = "rank_bm25 library"
except Exception:
    BM25_SRC = "NumPy fallback"
    class BM25Okapi:
        def __init__(self, corpus, k1=1.5, b=0.75):
            self.k1, self.b = k1, b
            self.tf = [Counter(d) for d in corpus]
            self.dl = np.array([len(d) for d in corpus]); self.avg = self.dl.mean(); self.N = len(corpus)
            df = Counter(t for d in corpus for t in set(d))
            self.idf = {t: math.log(1 + (self.N - n + 0.5)/(n + 0.5)) for t, n in df.items()}
            # invert: term -> list of (doc_id, freq) so scoring skips empty docs (fast at scale)
            self.post = {}
            for i, c in enumerate(self.tf):
                for t, f in c.items(): self.post.setdefault(t, []).append((i, f))
        def get_scores(self, query):
            sc = np.zeros(self.N)
            for t in query:
                if t not in self.idf: continue
                idf = self.idf[t]
                for i, f in self.post[t]:
                    denom = f + self.k1*(1 - self.b + self.b*self.dl[i]/self.avg)
                    sc[i] += idf * (f*(self.k1+1))/denom
            return sc

bm25 = BM25Okapi([t.split() for t in df_chunks["clean"]])
print("BM25 via:", BM25_SRC)

def retrieve_bm25(query, k=5):
    q = preprocess_text(query, **DEFAULT_PROFILE).split()
    sc = bm25.get_scores(q)
    idx = np.argsort(sc)[::-1][:k]
    return [(int(i), float(sc[i])) for i in idx if sc[i] > 0]

t0 = time.time(); bm25_summary = evaluate_retriever(retrieve_bm25); dt = time.time() - t0
print(f"BM25    { {k: round(v,3) for k,v in bm25_summary.items()} }  ({dt:.1f}s)")

---
# Part 5 — Dense Retrieval, Batched Encoding & FAISS

## 5.1 Encode once, cache to disk

### 🧠 Concept
Embedding thousands of chunks takes real time. Two scale techniques:
1. **Batched encoding** — the model processes chunks in batches (much faster than one-by-one).
2. **Disk caching** — save the embedding matrix with `np.save`; on later runs just `np.load` it.
   We guard the cache with the corpus size so a changed corpus triggers a re-encode.

In [ ]:
EMBEDDINGS_OK = True
try:
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)

    def build_embeddings():
        t0 = time.time()
        emb = embed_model.encode(df_chunks["text"].tolist(), batch_size=64,
                                 normalize_embeddings=True, show_progress_bar=True)
        print(f"Encoded {len(emb)} chunks in {time.time()-t0:.1f}s")
        return np.asarray(emb, dtype="float32")

    if EMB_CACHE.exists():
        doc_embeddings = np.load(EMB_CACHE)
        if len(doc_embeddings) != len(df_chunks):     # corpus changed → re-encode
            doc_embeddings = build_embeddings(); np.save(EMB_CACHE, doc_embeddings)
        else:
            print(f"Loaded cached embeddings {doc_embeddings.shape} from {EMB_CACHE}")
    else:
        doc_embeddings = build_embeddings(); np.save(EMB_CACHE, doc_embeddings)
        print(f"Cached embeddings → {EMB_CACHE}")
except Exception as e:
    EMBEDDINGS_OK = False
    print(f"⚠️  Embeddings unavailable ({type(e).__name__}). Dense/hybrid/FAISS/rerank will be skipped.")

In [ ]:
if EMBEDDINGS_OK:
    def retrieve_semantic(query, k=5):
        q = embed_model.encode([query], normalize_embeddings=True)
        sc = cosine_similarity(q, doc_embeddings).flatten()
        idx = sc.argsort()[::-1][:k]
        return [(int(i), float(sc[i])) for i in idx if sc[i] > 0]
    sem_summary = evaluate_retriever(retrieve_semantic)
    print(f"Embeddings { {k: round(v,3) for k,v in sem_summary.items()} }")
    # paraphrase win
    print('\nParaphrase "medicine to lower cholesterol":')
    for idx, s in retrieve_semantic("medicine to lower cholesterol", k=3):
        print(f"  {s:.3f}  {df_chunks.loc[idx,'drug']:<22s} {df_chunks.loc[idx,'field']}")
else:
    print("Skipped (embeddings unavailable).")

## 5.2 FAISS — brute force vs approximate (IVF)

### 🧠 Concept
At thousands of vectors, comparing the query against *every* chunk (brute force) starts to cost.
**FAISS** gives two options:
- **`IndexFlatIP`** — exact brute force (100% recall, slower as N grows).
- **`IndexIVFFlat`** — **approximate**: it clusters vectors into `nlist` cells and only searches the
  few nearest cells (`nprobe`). Massively faster at millions of vectors, at the cost of *maybe*
  missing a few results. This speed-vs-recall trade-off is *the* core idea of large-scale vector search.

We build both and compare their top-k on the same query to see they mostly agree.

In [ ]:
if EMBEDDINGS_OK:
    try:
        import faiss
        dim = doc_embeddings.shape[1]
        flat = faiss.IndexFlatIP(dim); flat.add(doc_embeddings)

        nlist = max(8, int(math.sqrt(len(doc_embeddings))))   # rule of thumb
        quant = faiss.IndexFlatIP(dim)
        ivf = faiss.IndexIVFFlat(quant, dim, nlist, faiss.METRIC_INNER_PRODUCT)
        ivf.train(doc_embeddings); ivf.add(doc_embeddings); ivf.nprobe = 8

        q = embed_model.encode(["how does metformin lower blood sugar"], normalize_embeddings=True).astype("float32")
        _, I_flat = flat.search(q, 5)
        _, I_ivf  = ivf.search(q, 5)
        overlap = len(set(I_flat[0]) & set(I_ivf[0]))
        print(f"FAISS built: Flat(exact) + IVF(nlist={nlist}, nprobe={ivf.nprobe})")
        print(f"Top-5 agreement between exact and approximate: {overlap}/5")
        print("Exact  top-5 drugs:", [df_chunks.loc[i,'drug'] for i in I_flat[0]])
        print("Approx top-5 drugs:", [df_chunks.loc[i,'drug'] for i in I_ivf[0]])
    except Exception as e:
        print(f"FAISS unavailable ({type(e).__name__}) — brute-force cosine still works.")
else:
    print("Skipped (embeddings unavailable).")

---
# Part 6 — Hybrid Retrieval + Cross-Encoder Reranking

## 6.1 Hybrid (BM25 + embeddings)

### 🧠 Concept
Same as the Learning Edition: min-max normalize each score to `[0,1]` and take a weighted sum
(α = 0.6, lexical-leaning because drug names are exact tokens). Sparse and dense fail in opposite
cases, so the blend is more robust than either alone.

In [ ]:
ALPHA = 0.6
def minmax(a):
    mn, mx = a.min(), a.max()
    return np.ones_like(a)*0.5 if mx-mn < 1e-12 else (a-mn)/(mx-mn)

if EMBEDDINGS_OK:
    def retrieve_hybrid(query, k=5):
        b = minmax(bm25.get_scores(preprocess_text(query, **DEFAULT_PROFILE).split()))
        q = embed_model.encode([query], normalize_embeddings=True)
        e = minmax(cosine_similarity(q, doc_embeddings).flatten())
        h = ALPHA*b + (1-ALPHA)*e
        idx = h.argsort()[::-1][:k]
        return [(int(i), float(h[i])) for i in idx]
    hybrid_summary = evaluate_retriever(retrieve_hybrid)
    print(f"Hybrid  { {k: round(v,3) for k,v in hybrid_summary.items()} }")
else:
    retrieve_hybrid, hybrid_summary = retrieve_bm25, bm25_summary
    print("Embeddings offline → hybrid falls back to BM25.")

## 6.2 Reranking — cast a wide net, then sharpen

### 🧠 Concept
This is the marquee scale upgrade. Fast retrievers (BM25/embeddings) are good at **recall** — getting
the right chunk *somewhere* in the top-50 — but imperfect at **ordering** the very top. A
**cross-encoder reranker** reads the *(query, chunk)* pair *together* (not as separate vectors) and
scores true relevance. It's too slow to run on all chunks, so the pattern is:

> **retrieve top-50 cheaply → rerank those 50 with the cross-encoder → keep the best 5.**

This "retrieve-then-rerank" cascade is how production RAG gets both speed *and* precision.

In [ ]:
RERANK_OK = False
if EMBEDDINGS_OK:
    try:
        from sentence_transformers import CrossEncoder
        reranker = CrossEncoder(RERANK_MODEL)
        RERANK_OK = True

        def retrieve_reranked(query, k=5, pool=RETRIEVE_POOL):
            candidates = retrieve_hybrid(query, k=pool)          # wide net
            pairs = [(query, df_chunks.loc[idx, "text"]) for idx, _ in candidates]
            scores = reranker.predict(pairs)                     # joint (query, chunk) scoring
            order = np.argsort(scores)[::-1][:k]
            return [(candidates[i][0], float(scores[i])) for i in order]

        rerank_summary = evaluate_retriever(retrieve_reranked)
        print(f"Reranked { {k: round(v,3) for k,v in rerank_summary.items()} }  (top-{RETRIEVE_POOL} → rerank → top-{K})")
    except Exception as e:
        print(f"⚠️  Reranker unavailable ({type(e).__name__}). Using hybrid as final retriever.")

# Choose the best available retriever as the RAG backend
if RERANK_OK:       final_retriever = retrieve_reranked
elif EMBEDDINGS_OK: final_retriever = retrieve_hybrid
else:               final_retriever = retrieve_bm25
print("RAG backend →", final_retriever.__name__)

## 6.3 Retrieval scoreboard

### 🧠 Concept
Line up everything on the same auto-generated eval set. Watch the progression: BM25 → embeddings →
hybrid → reranked usually climbs in MRR, showing each scale technique earning its keep.

In [ ]:
rows = [{"Retriever": "TF-IDF", **tfidf_summary}, {"Retriever": "BM25", **bm25_summary}]
if EMBEDDINGS_OK:
    rows += [{"Retriever": "Embeddings", **sem_summary}, {"Retriever": "Hybrid", **hybrid_summary}]
if RERANK_OK:
    rows += [{"Retriever": "Hybrid+Rerank", **rerank_summary}]
board = pd.DataFrame(rows).set_index("Retriever").round(3)
print(board.to_string())
print(f"\nBest MRR: {board['MRR'].idxmax()} ({board['MRR'].max():.3f})")

---
# Part 7 — RAG Assembly & Grounded Generation

## 7.1 Pack context + the grounded prompt (unchanged contract)

### 🧠 Concept
Identical to the Learning Edition — retrieval got fancier, but the *contract* with the LLM is the
same: numbered, citable sources + strict rules (use only the context, cite, keep numbers/negation,
add the disclaimer, refuse when unsupported).

In [ ]:
def pack_context(results):
    parts = []
    for rank, (idx, score) in enumerate(results, 1):
        r = df_chunks.loc[idx]
        parts.append(f'[Source {rank}] (drug={r["drug"]}, field={r["field"]}, score={score:.3f})\n{r["text"]}')
    return "\n\n".join(parts)

SYSTEM_PROMPT = """You are a pharmaceutical information assistant. Answer drug-related questions
using ONLY the provided context sources.

RULES:
1. Base your answer STRICTLY on the provided context. Do not use outside knowledge.
2. Cite sources inline using [Source N].
3. If the context does not contain enough information, say so explicitly.
4. NEVER provide medical advice. Always include: "This is not medical advice.
   Consult a healthcare professional for medical decisions."
5. Preserve exact numbers (dosages, frequencies) and negation words (no, not, never).
6. If the question cannot be answered from the context, refuse with an explanation."""

def build_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

---
CONTEXT:
{context}
---

QUESTION: {query}

ANSWER (include [Source N] citations and the medical disclaimer):"""

demo_q = QUERY_SPECS[0]["query"]
print("Assembled context preview for:", demo_q, "\n")
print(pack_context(final_retriever(demo_q, k=3))[:700], "...")

## 7.2 Generate with a local LLM (Ollama)

### 🧠 Concept
Same local, private, deterministic (`temperature=0`) generation. The full pipeline is still five
lines — the scale work all happened upstream in retrieval.

In [ ]:
import requests
def ollama_ok():
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5); r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        return any(OLLAMA_MODEL in m for m in models)
    except Exception as e:
        print("Ollama not reachable:", e); return False
OLLAMA_OK = ollama_ok()
print("Ollama available:", OLLAMA_OK)

def ask_ollama(prompt, temperature=0.0, timeout=180):
    r = requests.post(f"{OLLAMA_HOST}/api/generate",
                      json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
                            "options": {"temperature": temperature}}, timeout=timeout)
    r.raise_for_status(); return r.json()["response"]

def extract_answer(raw):
    return re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip() or raw.strip()

def rag_answer(query, k=K):
    results = final_retriever(query, k=k)
    raw = ask_ollama(build_prompt(query, pack_context(results)))
    return extract_answer(raw), results

In [ ]:
DEMOS = [
    QUERY_SPECS[0]["query"],
    "What is a medicine used to lower cholesterol and how does it work?",
    "Which drugs should be avoided during pregnancy?",
]
if OLLAMA_OK:
    for i, q in enumerate(DEMOS, 1):
        print("="*78, f"\nQ{i}: {q}\n" + "="*78)
        ans, results = rag_answer(q)
        for idx, s in results:
            print(f"  [{idx}] {s:.3f}  {df_chunks.loc[idx,'drug']:<20s} {df_chunks.loc[idx,'field']}")
        print("\n" + ans + "\n")
else:
    print("Ollama offline → showing retrieved context instead of a generated answer.\n")
    print(pack_context(final_retriever(DEMOS[0], k=3)))

---
# Part 8 — Evaluation, Safety & Scale Summary

## 8.1 Groundedness + the refusal test

### 🧠 Concept
Unchanged and non-negotiable: answers must **cite sources** and carry the **disclaimer**, and the
system must **refuse** questions the corpus can't answer (openFDA labels have no pricing) instead of
inventing facts. Safety is judged by refusals, at any scale.

In [ ]:
if OLLAMA_OK:
    ans, _ = rag_answer(DEMOS[0])
    print("Citations:", len(re.findall(r"\[Source \d+\]", ans)),
          "| Disclaimer:", "not medical advice" in ans.lower())

unanswerable = "What is the retail price of this drug in Egyptian pounds?"
print(f'\nRefusal test — "{unanswerable}"')
print("Nearest chunks (retrieval always returns something):")
for idx, s in final_retriever(unanswerable, k=3):
    print(f"  [{idx}] {s:.3f}  {df_chunks.loc[idx,'text'][:70]}")
if OLLAMA_OK:
    ans, _ = rag_answer(unanswerable)
    print("\nAnswer:\n", ans)
else:
    print("\nExpected: the model should state the context has no pricing information.")

## 8.2 Scale summary & cheat sheet

Same mental model as the Learning Edition — the pipeline shape never changes. What changes at scale is
the **engineering** around each box:

```
CHUNK      → sliding windows w/ overlap (long real text ≠ one tidy sentence)
CLEAN      → still: keep numbers + negation (safety > tidiness)
RETRIEVE   → sparse + dense + hybrid, then RERANK the top-N cascade
INDEX      → FAISS: exact (Flat) vs approximate (IVF) — speed/recall trade-off
CACHE      → download data once, encode embeddings once, np.save/np.load
EVALUATE   → auto-generate ground truth; measure P/R/Hit/MRR + TIMING
GENERATE   → same grounded prompt: cite, keep numbers, and REFUSE when unsupported
```

**The five scale lessons**
1. **Chunking dominates quality** — overlapping windows beat whole-field blobs on real text.
2. **Cache everything expensive** — data and embeddings are computed once, reused forever.
3. **Approximate search (IVF) trades a little recall for a lot of speed** — essential past ~10⁵ vectors.
4. **Retrieve-then-rerank** buys production-grade precision without re-scoring the whole corpus.
5. Safety is invariant to scale — **cite, preserve numbers/negation, and refuse.**

In [ ]:
print("="*66)
print("  PhARMA RAG — SCALE EDITION — COMPLETE")
print("="*66)
print(f"Drugs downloaded : {len(drugs_data)}  (openFDA, cached at {DRUGS_LARGE.name})")
print(f"Chunks           : {len(df_chunks)}  (sliding window {CHUNK_WORDS}w / {CHUNK_OVERLAP}w overlap)")
print(f"Eval queries     : {len(QUERY_SPECS)} auto-generated")
print(f"Retrievers       : TF-IDF, BM25 ({BM25_SRC})"
      + (", Embeddings, Hybrid" if EMBEDDINGS_OK else "")
      + (", +Reranker" if RERANK_OK else ""))
print(f"Vector index     : FAISS Flat + IVF" if EMBEDDINGS_OK else "Vector index : (embeddings offline)")
print(f"LLM              : {OLLAMA_MODEL} ({'connected' if OLLAMA_OK else 'offline'})")
print("\nTweak TARGET_DRUGS, CHUNK_WORDS, ALPHA, RETRIEVE_POOL, or ivf.nprobe and re-run to feel the trade-offs.")